---
title: Performance comparison (S3)
---

# Aggregating ICESat-2 elevations via S3 direct access

This is the S3 version of the [HTTPS performance
notebook](./02-performance.ipynb). S3 direct access avoids the HTTPS
proxy and connects to NASA's S3 buckets directly, which requires
running in **us-west-2** (e.g., on an EC2 instance or Lambda).

The workflow is the same: read ATL06 `h_li`, `h_li_sigma`, `latitude`,
`longitude`, and `atl06_quality_summary` from all 6 beams, filter by
quality, and compute weighted-mean elevation in 1-degree latitude bins.

1. **h5coro** (sequential, S3 driver) -- how magg reads data today
2. **async-hdf5 + zarrs** (concurrent, S3Store) -- Rust I/O + Rust codecs

In [ ]:
import warnings

warnings.filterwarnings("ignore", message="Numcodecs codecs are not in the Zarr")
warnings.filterwarnings(
    "ignore", category=UserWarning, message=".*does not have a Zarr V3 specification.*"
)
warnings.filterwarnings("ignore", category=FutureWarning, module="earthaccess")
warnings.filterwarnings("ignore", category=DeprecationWarning, module="tqdm")

## Find granules and get S3 credentials

In [ ]:
import asyncio
import time
from urllib.parse import urlparse

import earthaccess
import h5coro
import numpy as np
import zarr
from async_hdf5.store import S3Store
from async_hdf5.zarr import open_hdf5
from h5coro import s3driver

auth = earthaccess.login()

results = earthaccess.search_data(
    short_name="ATL06",
    cloud_hosted=True,
    bounding_box=(-120, -76, -95, -73),  # Amundsen Sea, West Antarctica
    temporal=("2024-01-01", "2024-03-31"),
    count=20,
)

# Get S3 URLs (direct access)
s3_urls = []
for granule in results:
    links = granule.data_links(access="direct")
    h5 = [l for l in links if l.endswith(".h5")]
    if h5:
        s3_urls.append(h5[0])

# Get temporary S3 credentials for this collection
s3_creds = earthaccess.get_s3_credentials(results=results)
print(f"{len(s3_urls)} granules over Amundsen Sea sector")
print(f"Example: {s3_urls[0]}")

## Shared helpers

In [ ]:
BEAMS = ["gt1l", "gt1r", "gt2l", "gt2r", "gt3l", "gt3r"]


def is_valid(h_li, sigma, lat, quality):
    """magg's quality + validity filter for ATL06."""
    return (
        (quality == 0)
        & np.isfinite(h_li)
        & (h_li < 1e10)
        & np.isfinite(sigma)
        & (sigma > 0)
        & np.isfinite(lat)
    )


def aggregate(all_lat, all_h_li, all_sigma):
    """Weighted-mean elevation in 1-degree latitude bands."""
    lat_bins = np.arange(-80, -70, 1.0)
    bin_idx = np.digitize(all_lat, lat_bins) - 1
    means = np.full(len(lat_bins) - 1, np.nan)
    counts = np.zeros(len(lat_bins) - 1, dtype=int)
    for i in range(len(lat_bins) - 1):
        mask = bin_idx == i
        if mask.any():
            w = 1.0 / all_sigma[mask] ** 2
            means[i] = np.average(all_h_li[mask], weights=w)
            counts[i] = mask.sum()
    return lat_bins, means, counts

## 1. The baseline: h5coro (sequential, S3 driver)

h5coro's S3 driver reads directly from the bucket using AWS
credentials. URLs are passed as `bucket/key` (without the `s3://`
prefix).

In [ ]:
h5coro_credentials = {
    "aws_access_key_id": s3_creds["accessKeyId"],
    "aws_secret_access_key": s3_creds["secretAccessKey"],
    "aws_session_token": s3_creds["sessionToken"],
}

t0 = time.perf_counter()
all_lat_h5coro = []
all_hli_h5coro = []
all_sigma_h5coro = []

for s3_url in s3_urls:
    # h5coro expects bucket/key without s3:// prefix
    resource = s3_url.replace("s3://", "", 1)
    h5obj = h5coro.H5Coro(resource, s3driver.S3Driver, credentials=h5coro_credentials)

    for beam in BEAMS:
        prefix = f"/{beam}/land_ice_segments"
        datasets = [
            f"{prefix}/h_li",
            f"{prefix}/h_li_sigma",
            f"{prefix}/latitude",
            f"{prefix}/longitude",
            f"{prefix}/atl06_quality_summary",
        ]
        try:
            result = h5obj.readDatasets(datasets=datasets, block=True)
        except Exception:
            continue

        h_li = np.asarray(result[f"{prefix}/h_li"])
        sigma = np.asarray(result[f"{prefix}/h_li_sigma"])
        lat = np.asarray(result[f"{prefix}/latitude"])
        quality = np.asarray(result[f"{prefix}/atl06_quality_summary"])

        valid = is_valid(h_li, sigma, lat, quality)
        all_lat_h5coro.append(lat[valid])
        all_hli_h5coro.append(h_li[valid])
        all_sigma_h5coro.append(sigma[valid])

    h5obj.close()

all_lat_h5coro = np.concatenate(all_lat_h5coro)
all_hli_h5coro = np.concatenate(all_hli_h5coro)
all_sigma_h5coro = np.concatenate(all_sigma_h5coro)
lat_bins, means_h5coro, counts_h5coro = aggregate(
    all_lat_h5coro, all_hli_h5coro, all_sigma_h5coro
)

elapsed_h5coro = time.perf_counter() - t0
print(
    f"h5coro sequential (S3): {elapsed_h5coro:.2f}s for {len(s3_urls)} granules x {len(BEAMS)} beams"
)
print(f"Total valid points: {len(all_lat_h5coro):,}")

## 2. async-hdf5 + zarrs (concurrent, S3Store)

async-hdf5's S3Store connects directly to the bucket. We pass the
temporary credentials from earthaccess and the object key (everything
after `s3://bucket/`).

In [ ]:
zarr.config.set(
    {
        "async.concurrency": 128,
        "codec_pipeline.path": "zarrs.ZarrsCodecPipeline",
    }
)

# Parse bucket from first URL; all granules are in the same bucket
parsed = urlparse(s3_urls[0])
bucket = parsed.netloc

s3_store = S3Store(
    bucket=bucket,
    region="us-west-2",
    access_key_id=s3_creds["accessKeyId"],
    secret_access_key=s3_creds["secretAccessKey"],
    token=s3_creds["sessionToken"],
)


async def read_beam(s3_url, beam):
    """Read one beam from one granule via S3; return valid (lat, h_li, sigma)."""
    key = urlparse(s3_url).path.lstrip("/")
    try:
        hdf5_store = await open_hdf5(
            path=key,
            store=s3_store,
            group=f"{beam}/land_ice_segments",
        )
    except Exception:
        return None
    group = zarr.open_group(hdf5_store, mode="r")
    h_li = group["h_li"][:]
    sigma = group["h_li_sigma"][:]
    lat = group["latitude"][:]
    quality = group["atl06_quality_summary"][:]

    valid = is_valid(h_li, sigma, lat, quality)
    if not valid.any():
        return None
    return lat[valid], h_li[valid], sigma[valid]


t0 = time.perf_counter()
tasks = [read_beam(u, b) for u in s3_urls for b in BEAMS]
results_async = await asyncio.gather(*tasks)
results_async = [r for r in results_async if r is not None]

all_lat_async = np.concatenate([r[0] for r in results_async])
all_hli_async = np.concatenate([r[1] for r in results_async])
all_sigma_async = np.concatenate([r[2] for r in results_async])
lat_bins, means_async, counts_async = aggregate(
    all_lat_async, all_hli_async, all_sigma_async
)

elapsed_async = time.perf_counter() - t0
print(
    f"async-hdf5 + zarrs (S3): {elapsed_async:.2f}s for {len(s3_urls)} granules x {len(BEAMS)} beams"
)
print(f"Total valid points: {len(all_lat_async):,}")

## Results

In [ ]:
import pandas as pd

n_reads = len(s3_urls) * len(BEAMS)
speedup = elapsed_h5coro / elapsed_async

summary = pd.DataFrame(
    {
        "Library": [
            "h5coro (sequential, S3)",
            "async-hdf5 + zarrs (concurrent, S3)",
        ],
        f"Time ({n_reads} beam reads)": [
            f"{elapsed_h5coro:.1f}s",
            f"{elapsed_async:.1f}s",
        ],
        "Valid points": [
            f"{len(all_lat_h5coro):,}",
            f"{len(all_lat_async):,}",
        ],
        "Speedup": ["baseline", f"{speedup:.1f}x"],
    }
)
summary = summary.set_index("Library")
summary

## Aggregated elevation profile

In [ ]:
import matplotlib.pyplot as plt

bin_centers = (lat_bins[:-1] + lat_bins[1:]) / 2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

valid_bins = ~np.isnan(means_async)
ax1.plot(bin_centers[valid_bins], means_h5coro[valid_bins], "o-", label="h5coro (S3)")
ax1.plot(
    bin_centers[valid_bins],
    means_async[valid_bins],
    "s--",
    label="async-hdf5 + zarrs (S3)",
)
ax1.set_xlabel("Latitude (deg)")
ax1.set_ylabel("Weighted mean elevation (m)")
ax1.set_title("Ice sheet elevation by latitude (Amundsen Sea)")
ax1.legend()

ax2.bar(bin_centers[valid_bins], counts_async[valid_bins], width=0.8, alpha=0.7)
ax2.set_xlabel("Latitude (deg)")
ax2.set_ylabel("Number of valid points")
ax2.set_title("Data density by latitude")

fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
libraries = ["h5coro\n(sequential, S3)", "async-hdf5 + zarrs\n(concurrent, S3)"]
times = [elapsed_h5coro, elapsed_async]
colors = ["#e74c3c", "#2ecc71"]
bars = ax.bar(
    libraries, times, color=colors, width=0.4, edgecolor="black", linewidth=0.5
)

for bar, t in zip(bars, times):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(times) * 0.02,
        f"{t:.1f}s",
        ha="center",
        fontweight="bold",
    )

ax.set_ylabel("Wall-clock time (s)")
ax.set_title(f"Aggregation time: {len(s3_urls)} granules x {len(BEAMS)} beams (S3)")
ax.set_ylim(0, max(times) * 1.15)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()

## HTTPS vs S3

The [HTTPS notebook](./02-performance.ipynb) works from anywhere but
goes through NASA's HTTPS proxy. This notebook uses S3 direct access,
which requires running in us-west-2 but avoids the proxy overhead.
For magg's Lambda workers (which already run in us-west-2), S3 is the
natural choice.